In [2]:
from __future__ import annotations
from typing import Tuple
import torch
from torch import Tensor
import torch.nn as nn
import torch.nn.functional as F
import torchaudio


In [7]:
from pathlib import Path

wav_files = list(Path("/content/drive/MyDrive").glob("*.wav"))
print(wav_files)

[PosixPath('/content/drive/MyDrive/nearend_mic_fileid_0.wav'), PosixPath('/content/drive/MyDrive/nearend_speech_fileid_0.wav')]


This code is used to mount google drive to server(e.g, GPU T4)

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


ERB filter bank

In [9]:
def load_mono(path):
    wav, sr = torchaudio.load(path)    # sr: sample rate
    wav = wav.mean(dim=0)
    return wav, sr

def stft_magnitude(wav, n_fft=512, hop_length=128, win_length=512):
    window = torch.hann_window(win_length)

    spec = torch.stft(
        wav,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
        center=True,
        return_complex=True,
    )

    # [freq, time] -> [time, freq]
    return spec.abs().transpose(0, 1)

def hz_to_erb(freq):
    return 21.4 * torch.log10(1.0 + 0.00437 * freq)

def erb_to_hz(erb):
    return (10 ** (erb / 21.4) - 1.0) / 0.00437

def make_erb_filterbank(sample_rate, n_fft, erb_bins, low_freq=0.0, high_freq=None):
    if high_freq is None:
        high_freq = sample_rate / 2

    erb_edges = torch.linspace(
        hz_to_erb(torch.tensor(low_freq)),
        hz_to_erb(torch.tensor(high_freq)),
        erb_bins + 2,
    )

    hz_edges = erb_to_hz(erb_edges)
    fft_freqs = torch.linspace(0.0, sample_rate / 2, n_fft // 2 + 1)

    filterbank = torch.zeros(erb_bins, n_fft // 2 + 1)

    for i in range(erb_bins):
        left = hz_edges[i]
        center = hz_edges[i + 1]
        right = hz_edges[i + 2]

        rising = (fft_freqs - left) / (center - left)
        falling = (right - fft_freqs) / (right - center)

        filterbank[i] = torch.minimum(rising, falling).clamp_min(0.0)
    return filterbank

def wav_to_erb(path, sample_rate, filterbank, n_fft=512, hop_length=128, win_length=512):
    wav, sr = load_mono(path)

    if sr != sample_rate:
        wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=sample_rate)

    magnitude = stft_magnitude(wav, n_fft=n_fft, hop_length=hop_length, win_length=win_length)
    erb = magnitude @ filterbank.T
    erb = torch.log1p(erb)
    return erb 


Set up layers

In [19]:
def channel_shuffle(x: Tensor, groups: int) -> Tensor:
    b, t, d = x.shape
    if d % groups != 0:
        raise ValueError("Feature dimension must be divisible by groups")

    features_per_group = d // groups
    x = x.view(b, t, groups, features_per_group)
    x = x.transpose(2, 3).contiguous()
    x = x.view(b, t, d)

    return x

class SeparableConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, *, kernel_size: Tuple[int, int] = (3, 2), stride: Tuple[int, int] = (1, 1)) -> None:
        super().__init__()

        kt, kf = kernel_size
        self.pad = (
            kf // 2, # left
            kf - 1 - kf // 2, # right
            kt // 2, # top
            kt // 2, # bottom
        )

        self.depthwise = nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=0,
            groups=in_channels,
            bias=False
        )

        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False,
        )

        self.norm = nn.BatchNorm2d(out_channels)
        self.act = nn.ReLU()
    def forward(self, x: Tensor) -> Tensor:
        x = F.pad(x, self.pad)
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.norm(x)
        x = self.act(x)
        return x
    
class SeparableTConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, *, scale_factor: int = 2) -> None:
        super().__init__()
        self.scale_factor = scale_factor

        self.conv = SeparableConv2d(in_channels, out_channels, kernel_size=(3, 2))

    def forward(self, x: Tensor) -> Tensor:
        x = F.interpolate(
            x, 
            scale_factor=(1, self.scale_factor), # [width, height] ~ [time, frequency]
            mode="nearest"
        )
        return self.conv(x)

class GroupedLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, groups: int = 8, shuffle: bool = True) -> None:
        super().__init__()

        if in_features % groups:
            raise ValueError("in_features must be divisible by groups")

        if out_features % groups:
            raise ValueError("out_features must be divisible by groups")

        self.groups = groups
        self.shuffle = shuffle

        self.in_per_group = in_features // groups
        self.out_per_group = out_features // groups

        self.layers = nn.ModuleList(
            [
                nn.Linear(
                    self.in_per_group,
                    self.out_per_group,
                )
                for _ in range(groups)
            ]
        )

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, T, D]

        chunks = x.split(self.in_per_group, dim=-1)

        output_chunks = []

        for layer, chunk in zip(self.layers, chunks):
            y = layer(chunk)
            output_chunks.append(y)

        x = torch.cat(output_chunks, dim=-1)

        if self.shuffle:
            x = channel_shuffle(x, self.groups)

        return x

class GroupedGRU(nn.Module):
    def __init__(self, input_size: int = 512, hidden_size: int = 512, groups: int = 8, shuffle: bool = True) -> None:
        super().__init__()
        if input_size % groups:
            raise ValueError("input_size must be divisible by groups")
        if hidden_size % groups:
            raise ValueError("hidden_size must be divisible by groups")

        self.groups = groups
        self.shuffle = shuffle

        self.input_per_group = input_size // groups
        self.hidden_per_groups = hidden_size // groups

        self.grus = nn.ModuleList(
            [
                nn.GRU(
                    input_size=self.input_per_group,
                    hidden_size=self.hidden_per_groups,
                    batch_first=True,
                )
                for _ in range(groups)
            ]
        )


    def forward(self, x: Tensor) -> Tensor:
        # [B, T, D]
        chunks = x.split(self.input_per_group, dim=-1)
        outputs = []

        for gru, chunk in zip(self.grus, chunks):
            # GRU returns [output, hidden]
            y, _ = gru(chunk)
            outputs.append(y)

        x = torch.cat(outputs, dim=-1)

        if self.shuffle:
            x = channel_shuffle(x, self.groups)

        return x

class GroupedGRUStack(nn.Module):
    def __init__(self, size: int = 512, groups: int = 8, num_layers: int = 3) -> None:
        super().__init__()

        self.layers = nn.ModuleList(
            [
                GroupedGRU(
                    input_size=size,
                    hidden_size=size,
                    groups=groups,
                    shuffle=True,
                )
                for _ in range(num_layers)
            ]
        )

    def forward(self, x: Tensor) -> Tensor:
        for layer in self.layers:
            x = layer(x)

        return x

class PConv(nn.Module):
    def __init__(self, channels: int = 64) -> None:
        super().__init__()

        self.conv = nn.Conv2d(channels, channels, kernel_size=1, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        return self.conv(x)

Implement ERB encoder, decoder and deep filter net

In [25]:
class ERBEncoder(nn.Module):
    def __init__(self, erb_bins: int = 32, channels: int = 64, hidden_size: int = 512, groups: int = 8) -> None:
        super().__init__()

        if erb_bins % 8:
            raise ValueError("erb_bins must be divisible by 8")

        self.erb_bins = erb_bins
        self.channels = channels
        self.conv0 = SeparableConv2d(1, channels)
        self.conv1 = SeparableConv2d(channels, channels, stride=(1, 2)) # B -> B / 2
        self.conv2 = SeparableConv2d(channels, channels, stride=(1, 2))
        self.conv3 = SeparableConv2d(channels, channels, stride=(1, 2))
        self.glinear = GroupedLinear(in_features=channels*erb_bins//8, out_features=hidden_size, groups=groups)
        self.gru = GroupedGRUStack(size=hidden_size, groups=groups, num_layers=3)

    def forward(self, x: Tensor) -> Tuple[Tensor, Tensor, Tensor, Tensor, Tensor]:
        x0 = self.conv0(x)
        x1 = self.conv1(x0)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)

        batch, channels, time, freq = x3.shape
        x = x3.permute(0, 2, 1, 3)
        x = x.reshape(batch, time, channels*freq)

        x = self.glinear(x)
        embedding = self.gru(x)

        return x0, x1, x2, x3, embedding

class ERBDecoder(nn.Module):
    def __init__(self, channels: int = 64, erb_bins: int = 32, hidden_size: int = 512) -> None:
        super().__init__()

        bottleneck_freq = erb_bins // 8
        bottleneck_size = channels * bottleneck_freq

        self.channels = channels
        self.bottelneck_freq = bottleneck_freq

        self.linear = GroupedLinear(hidden_size, bottleneck_size)

        self.p3 = PConv(channels)
        self.p2 = PConv(channels)
        self.p1 = PConv(channels)
        self.p0 = PConv(channels)

        self.up3 = SeparableTConv2d(channels, channels)
        self.up2 = SeparableTConv2d(channels, channels)
        self.up1 = SeparableTConv2d(channels, channels)

        self.final_conv = SeparableConv2d(channels, 1, kernel_size=(3, 2))
        self.output_activation = nn.Sigmoid()

    def forward(self, embedding: Tensor, x0: Tensor, x1: Tensor, x2: Tensor, x3: Tensor) -> Tensor:
        batch, time, _ = embedding.shape

        x = self.linear(embedding)

        x = x.reshape(batch, time, self.channels, self.bottelneck_freq)
        x = x.permute(0, 2, 1, 3)

        x = x + self.p3(x3)

        x = self.up3(x)
        x = x + self.p2(x2)

        x = self.up2(x)
        x = x + self.p1(x1)

        x = self.up1(x)
        x = x + self.p0(x0)

        gains = self.final_conv(x)
        gains = self.output_activation(gains)

        return gains

# class DFNet(nn.Module):
#     def __init__(self, df_bins: int, df_order: int, channels: int=64, hidden_size: int = 512, groups: int=8) -> None:
#         super().__init__()

#         self.df_bins = df_bins
#         self.df_order = df_order
#         self.channels = channels

#         self.conv0 = SeparableConv2d(1, channels, stride=(1, 1))
#         self.conv1 = SeparableConv2d(channels, channels, stride=(1, 2))
#         self.projection = GroupedLinear(channels*(df_bins // 2), hidden_size, groups=groups)
#         self.grus = GroupedGRUStack(size=hidden_size, groups=groups, num_layers=2)
#         self.pconv = PConv(channels)
#         self.output = nn.Linear(hidden_size, df_bins*df_order*2)
#         self.merge = nn.Linear(hidden_size*2, hidden_size)

#     def forward(self, complex_features: Tensor, encoder_embedding: Tensor) -> Tensor:
#         x = self.conv0(complex_features)
#         x = self.conv1(x)

#         batch, channels, time, freq = x.shape
#         x = x.permute(0, 2, 1, 3)
#         x = x.reshape(batch, time, channels * freq)

#         x = self.projection(x)
#         x = self.grus(x)

#         x = torch.cat([x, encoder_embedding], dim=-1)
#         x = self.merge(x)

#         coefficients = self.output(x)

#         coefficients = coefficients.reshape(batch, time, self.df_bins, self.df_order, 2)

#         return coefficients

        

In [21]:
class DeepFilterNetDNN(nn.Module):
    """
    rb_features:
        [B, 1, T, B_erb]

    complex_features:
        [B, 1, T, F_df]

    erb_gains:
        [B, 1, T, B_erb]
    
    df_coefficients:
        [B, T, F_df, N, 2]
    """

    def __init__(self, erb_bins: int = 32, df_bins: int = 96, df_order: int = 5, channels: int = 64, hidden_size: int = 512, groups: int = 8) -> None:
        super().__init__()

        self.encoder = ERBEncoder(
            erb_bins=erb_bins,
            channels=channels,
            hidden_size=hidden_size,
            groups=groups
        )

        self.decoder = ERBDecoder(
            channels=channels,
            erb_bins=erb_bins,
            hidden_size=hidden_size
        )

        # self.df_net = DFNet(
        #     df_bins=df_bins,
        #     df_order=df_order,
        #     channels=channels,
        #     hidden_size=hidden_size,
        #     groups=groups,
        # )

    # def forward(self, erb_features: Tensor, complex_features: Tensor) -> Tuple[Tensor, Tensor]:
    #     e0, e1, e2, e3, embedding = self.encoder(erb_features)
    #     gains = self.decoder(embedding, e0, e1, e2, e3)
    #     #df_coefficients = self.df_net(complex_features, embedding)

    #     return gains, df_coefficients

    def forward(self, erb_features: Tensor) -> Tensor:
            e0, e1, e2, e3, embedding = self.encoder(erb_features)
            gains = self.decoder(embedding, e0, e1, e2, e3)
            #df_coefficients = self.df_net(complex_features, embedding)
            return gains

Check available GPU

In [27]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

True
Tesla T4


In [17]:
sample_rate = 16000
n_fft = 512
hop_length = 128
win_length = 512
erb_bins = 32

filterbank = make_erb_filterbank(
    sample_rate=sample_rate,
    n_fft=n_fft,
    erb_bins=erb_bins,
)

input_erb = wav_to_erb(
    "/content/drive/MyDrive/nearend_mic_fileid_0.wav",
    sample_rate,
    filterbank,
    n_fft,
    hop_length,
    win_length,
)

target_erb = wav_to_erb(
    "/content/drive/MyDrive/nearend_speech_fileid_0.wav",
    sample_rate,
    filterbank,
    n_fft,
    hop_length,
    win_length,
)

num_frames = min(input_erb.shape[0], target_erb.shape[0])

input_erb = input_erb[:num_frames]
target_erb = input_erb[:num_frames]

# Add batch and channel dimensions:
# [T, E] -> [B, 1, T, E]
input_features = input_erb.unsqueeze(0).unsqueeze(0)
target_features = target_erb.unsqueeze(0).unsqueeze(0)


target_mask = target_features / (input_features + 1e-8)
target_mask = target_mask.clamp(0.0, 1.0)


In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepFilterNetDNN(erb_bins=erb_bins, channels=64, hidden_size=512, groups=8).to(device)

model.eval()

input_features = input_features.to(device)
target_mask = target_mask.to(device)

with torch.no_grad():
    predicted_gains = model(input_features)


print("Input:", input_features.shape)
print("Predicted gains:", predicted_gains.shape)
print("Target mask:", target_mask.shape)
print(input_features)
print(predicted_gains)

Input: torch.Size([1, 1, 1250, 32])
Predicted gains: torch.Size([1, 1, 1250, 32])
Target mask: torch.Size([1, 1, 1250, 32])
tensor([[[[0.0659, 0.1323, 0.1620,  ..., 0.0071, 0.0042, 0.0103],
          [0.0964, 0.2222, 0.3713,  ..., 0.0072, 0.0045, 0.0114],
          [0.1481, 0.1760, 0.3388,  ..., 0.0073, 0.0055, 0.0143],
          ...,
          [0.2351, 0.4280, 0.4704,  ..., 0.0334, 0.1515, 0.3414],
          [0.2378, 0.1281, 0.4862,  ..., 0.0339, 0.1490, 0.3666],
          [0.3125, 0.5139, 0.6250,  ..., 0.0432, 0.1223, 0.2762]]]],
       device='cuda:0')
tensor([[[[0.5002, 0.5014, 0.5031,  ..., 0.5002, 0.5002, 0.5001],
          [0.5000, 0.5000, 0.5000,  ..., 0.5000, 0.5000, 0.5000],
          [0.5000, 0.5000, 0.5000,  ..., 0.5000, 0.5001, 0.5001],
          ...,
          [0.5000, 0.5000, 0.5000,  ..., 0.5000, 0.5000, 0.5000],
          [0.5000, 0.5000, 0.5000,  ..., 0.5000, 0.5000, 0.5000],
          [0.5000, 0.5000, 0.5000,  ..., 0.5000, 0.5000, 0.5000]]]],
       device='cuda:0')


In [33]:
wav, sr = load_mono("/content/drive/MyDrive/nearend_speech_fileid_0.wav")

print(wav)
print("Sample rate:", sr)
print("Number of samples:", wav.shape)
print("Duration:", wav.shape[0] / sr, "seconds")

magnitude = stft_magnitude(
    wav,
    n_fft=512,
    hop_length=128,
    win_length=512,
)

print("STFT shape:", magnitude.shape)

tensor([0., 0., 0.,  ..., 0., 0., 0.])
Sample rate: 16000
Number of samples: torch.Size([159999])
Duration: 9.9999375 seconds
STFT shape: torch.Size([1250, 257])
